In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Membuat Dataset: Matriks Rating (1-5)
# Baris = User, Kolom = Lagu Akustik/Story-driven
# NaN berarti user belum pernah mendengarkan/memberi rating lagu tersebut
data = {
    'Supermarket Flowers': [5, 4, np.nan, 1, 5],
    'Photograph': [4, 5, 2, np.nan, 4],
    'All of Me': [4, 5, np.nan, 2, 5],
    'Someone Like You': [np.nan, 4, 5, 1, 4],
    'Heavy Metal Rock': [1, 1, 5, 5, np.nan] # Lagu genre beda sebagai pembanding
}

users = ['User_A', 'User_B', 'User_C', 'User_D', 'User_Target']
df_ratings = pd.DataFrame(data, index=users)

print("=== USER-ITEM RATING MATRIX ===")
display(df_ratings)

# ANALISIS UNTUK DOSEN:
# Matriks ini menunjukkan 'Sparsity' (banyak nilai NaN).
# Kita ingin merekomendasikan lagu untuk 'User_Target' yang belum mendengar lagu 'Heavy Metal Rock'.

=== USER-ITEM RATING MATRIX ===


,Supermarket Flowers,Photograph,All of Me,Someone Like You,Heavy Metal Rock
User_A,5.0,4.0,4.0,NaN,1.0
User_B,4.0,5.0,5.0,4.0,1.0
User_C,NaN,2.0,NaN,5.0,5.0
User_D,1.0,NaN,2.0,1.0,5.0
User_Target,5.0,4.0,5.0,4.0,NaN


In [2]:
# 2. PREPROCESSING UNTUK MATEMATIKA
# Cosine Similarity tidak bisa menghitung nilai NaN.
# Solusi umum: Isi nilai kosong dengan angka 0 (mengindikasikan tidak ada interaksi).
# Atau bisa juga diisi dengan rata-rata rating user tersebut (Mean Centering).
# Kita gunakan fillna(0) untuk kesederhanaan formula dasar.

df_ratings_filled = df_ratings.fillna(0)

print("=== RATING MATRIX SETELAH PREPROCESSING ===")
display(df_ratings_filled)

=== RATING MATRIX SETELAH PREPROCESSING ===


,Supermarket Flowers,Photograph,All of Me,Someone Like You,Heavy Metal Rock
User_A,5.0,4.0,4.0,0.0,1.0
User_B,4.0,5.0,5.0,4.0,1.0
User_C,0.0,2.0,0.0,5.0,5.0
User_D,1.0,0.0,2.0,1.0,5.0
User_Target,5.0,4.0,5.0,4.0,0.0


In [3]:
# =======================================================
# 3. MENGHITUNG COSINE SIMILARITY ANTAR USER
# =======================================================

# Menghitung matriks kemiripan untuk semua user sekaligus
similarity_matrix = cosine_similarity(df_ratings_filled)

# Ubah ke DataFrame agar mudah dibaca
df_sim = pd.DataFrame(similarity_matrix, index=users, columns=users)

print("=== MATRIKS COSINE SIMILARITY ANTAR USER ===")
display(df_sim.round(3))

# =======================================================
# 4. PENALARAN REKOMENDASI (REASONING)
# =======================================================
target = 'User_Target'

# Ambil skor kemiripan User Target dengan user lain, urutkan dari yang paling mirip
similar_users = df_sim[target].drop(target).sort_values(ascending=False)
print(f"\nSkor Kemiripan dengan {target}:")
print(similar_users)

# User_Target paling mirip dengan User_A (skor 0.98) dan User_B (skor 0.96)
# Keduanya tidak menyukai 'Heavy Metal Rock' (rating 1).
# Kesimpulan Logis: Sistem TIDAK AKAN merekomendasikan 'Heavy Metal Rock' ke User_Target.

=== MATRIKS COSINE SIMILARITY ANTAR USER ===


,User_A,User_B,User_C,User_D,User_Target
User_A,1.000,0.879,0.232,0.424,0.885
User_B,0.879,1.000,0.523,0.453,0.982
User_C,0.232,0.523,1.000,0.733,0.421
User_D,0.424,0.453,0.733,1.000,0.377
User_Target,0.885,0.982,0.421,0.377,1.000



Skor Kemiripan dengan User_Target:
User_B    0.981836
User_A    0.884523
User_C    0.420779
User_D    0.376848
Name: User_Target, dtype: float64
